# Donut fine-tune - InBody **270 + 570** v4 · Kaggle runner

Retrains Donut on the **v4** both-device synthetic set (2500 inbody_270 + 2500 inbody_570).
v4 fixes the template geometry: sheets now render at aspect 0.702 and >=2352 px wide, so they
are *downscaled* onto Donut's 2560x1920 canvas the same way a real phone photo is. Every
dataset up to v3 was upscaled, which is why `donut-both-v3` met sharp text only at inference.
See ADR-0006 (2026-09-09 amendment) and ADR-0007.

Fresh from `naver-clova-ix/donut-base`, **5 epochs**, **both devices**. Do **not** resume from
`donut-both-v3` - the input distribution changed.

**Kaggle setup before running:**
1. *Settings -> Accelerator* -> **GPU T4 x2** (we pin to one GPU below - donut-base OOMs on GPU0 with both).
2. *Settings -> Internet* -> **On**.
3. *Add-ons -> Secrets* -> add **`GH_TOKEN`** (fine-grained, Contents: Read-only, scoped to QeekOw/InForm).
4. Upload `synth_v4_both.zip` to **Kaggle -> Datasets -> New Dataset**, then attach it here via
   **Add Input** (right panel). Cell 3 auto-detects + extracts it.

**Budget the 12 h session cap.** v3 took ~2.5-3 h for one epoch on 5000 sheets, so 5 epochs
does not fit in one commit. `train.py` uses `save_strategy="epoch"` with no
`save_total_limit`, so every epoch checkpoint survives; when a session is cut short, re-run
the resume cell (cell 5) in a fresh session pointed at the same output dir.


In [ ]:
# GPU + the two flags from prior runs: pin to one GPU (dual-T4 OOMs donut-base on
# GPU0) and enable expandable segments to avoid fragmentation OOMs.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the branch carrying the geometry fix + the real hold-out scorer.
BRANCH = 'feat/module-1-real-holdout-scorer'
from kaggle_secrets import UserSecretsClient
try:
    GH_TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
    REPO = f'https://{GH_TOKEN}@github.com/QeekOw/InForm.git'
except Exception:
    REPO = 'https://github.com/QeekOw/InForm.git'  # public fallback
%cd /kaggle/working
!rm -rf /kaggle/working/repo
!git clone --branch $BRANCH --single-branch $REPO repo
%cd /kaggle/working/repo
!pip install -q -e '.[training]' gdown

In [ ]:
# Dataset attached as a Kaggle Dataset (Add Input, right panel) - no Drive/gdown.
# Find the sheets wherever Kaggle mounted them; if the input is still a .zip,
# extract it to /kaggle/tmp. Sets DATA_DIR to the folder holding the sheets.
import glob, os, shutil

# generate_dataset writes .jpg (inform.training.dataset.IMAGE_SUFFIXES); runs
# before that wrote .png, so accept either rather than silently finding nothing.
SUFFIXES = ('jpg', 'jpeg', 'png')

def find_sheets(root):
    return sorted(p for s in SUFFIXES for p in glob.glob(f'{root}/**/*.{s}', recursive=True))

sheets = find_sheets('/kaggle/input')
if not sheets:
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No sheets or zip under /kaggle/input - click Add Input and attach your dataset.'
    shutil.unpack_archive(zips[0], '/kaggle/tmp')
    sheets = find_sheets('/kaggle/tmp')
assert sheets, 'No sheets found after extract - check the dataset contents.'
DATA_DIR = os.path.dirname(sheets[0])
n270 = len([p for p in sheets if 'inbody_270' in p]); n570 = len([p for p in sheets if 'inbody_570' in p])
print('DATA_DIR =', DATA_DIR, '| sheets:', len(sheets), '| 270:', n270, '570:', n570)


In [ ]:
# Fresh from donut-base, 5 epochs, BOTH devices. Do NOT resume from donut-both-v3:
# v4 changed the input distribution (aspect 0.949 -> 0.702, width 941-1251 -> >=2352,
# PNG -> JPEG), so the old weights are trained on a different-looking sheet.
# batch 1 + grad-accum 4 fits the 2560x1920 canvas on a T4; effective batch 4.
# --batch-size 2 OOMs even on 16 GB - raise grad-accum instead.
# 4 dataloader workers keep the GPU fed while JPEGs decode.
CHECKPOINT_DIR = '/kaggle/working/donut-both-v4'
!python -m inform.training.train   --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR   --model-name-or-path naver-clova-ix/donut-base   --epochs 5 --batch-size 1 --gradient-accumulation-steps 4   --dataloader-num-workers 4 --learning-rate 3e-5

In [ ]:
# RESUME ONLY - run this instead of cell 4 when a previous session hit the 12 h cap.
# Attach that session's output as an input, copy the checkpoints into CHECKPOINT_DIR,
# then --resume picks up from the last checkpoint-* subdir.
#
# import glob, shutil, os
# prev = glob.glob('/kaggle/input/**/donut-both-v4', recursive=True)[0]
# shutil.copytree(prev, CHECKPOINT_DIR, dirs_exist_ok=True)
# !python -m inform.training.train #   --data-dir "$DATA_DIR" --output-dir $CHECKPOINT_DIR #   --model-name-or-path naver-clova-ix/donut-base #   --epochs 5 --batch-size 1 --gradient-accumulation-steps 4 #   --dataloader-num-workers 4 --learning-rate 3e-5 --resume

In [ ]:
# Every epoch checkpoint is kept (save_strategy='epoch', save_total_limit=None), so
# /kaggle/working holds checkpoint-* subdirs plus the final top-level model. All of it
# lands in the Save Version output. ~2.4 GB per checkpoint - watch the 20 GB /kaggle/working
# limit, and download to a drive with room (NOT C:, which has ~12 GB free).
!du -sh /kaggle/working/donut-both-v4/* | sort -h
!ls -la /kaggle/working/donut-both-v4

## After the run

Score **every** epoch checkpoint, not just the last - ADR-0006 wants both the real and the
synthetic number, with the synthetic-vs-real gap stated explicitly.

**Real hand-labelled hold-out** (12 photos, `data/real_holdout/`, gitignored - local only):

```bash
python -m inform.holdout --data-dir data/real_holdout     --labels data/real_holdout/labels.json --donut-checkpoint <ckpt>
```

Baseline to beat (`donut-both-v3`, measured):

| | v3 |
|---|---|
| core fields | 33/36 (91.7%) |
| segmental lean | 23/30 (76.7%) |
| critical (LBM + limbs) | 26/36 (72.2%) |
| outcome split (n=12) | 3 usable / 3 flagged / 0 unread / 6 refused |

**The sharpest single indicator is not an aggregate**: check whether the **sheet_05 panel
crossing** disappears. That sheet scored *usable* with every cross-check passing while its
right-arm and right-leg lean values were the adjacent Segmental Fat panel's left-side values.
If geometry was the cause, that specific failure goes away.

**Held-out synthetic (both devices):**

```bash
python -m inform.compare --data-dir D:/cera/data/holdout_v4_both     --donut-checkpoint <ckpt> --skip-vlm
```

Segmental figures are only comparable post-ADR-0006-amendment (1% relative bound as well as
+-0.1 absolute). 570 still has no real phone photo - ship it synthetic-validated.